# LSTMによるプレー中外判定モデルのトレーニング

このノートブックは、卓球のプレー中/プレー外を判別するLSTMモデルをGoogle Colab上でトレーニングするためのものです。

## セットアップ手順
1. Google Driveにデータをアップロード
2. Colabでこのノートブックを開く
3. ランタイムタイプをGPUに設定
4. セルを順番に実行

## 1. Google Driveのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 必要なライブラリのインストール

In [ ]:
# 必要に応じて追加のライブラリをインストール
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm tensorboard

## 3. プロジェクトのクローン/コピー

Google Drive上のプロジェクトフォルダへのパスを指定してください

In [ ]:
import os
import sys

# プロジェクトのパスを指定（例: /content/drive/MyDrive/Visuable_for_you_tabletennis）
PROJECT_PATH = '/content/drive/MyDrive/Visuable_for_you_tabletennis'

# プロジェクトが存在しない場合はGitHubからクローン（オプション）
if not os.path.exists(PROJECT_PATH):
    print(f"プロジェクトが見つかりません: {PROJECT_PATH}")
    print("GitHubからクローンする場合は以下のコメントを外してください")
    # !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git {PROJECT_PATH}
else:
    print(f"プロジェクトを使用: {PROJECT_PATH}")

# プロジェクトパスをPythonのパスに追加
sys.path.insert(0, PROJECT_PATH)

# 作業ディレクトリを変更
os.chdir(PROJECT_PATH)
print(f"作業ディレクトリ: {os.getcwd()}")

## 4. 必要なモジュールのインポート

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import json
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# プロジェクトのモジュールをインポート
from src.models.play_classifier import PlayClassifierLSTM, PlayClassifierCNNLSTM
from src.dataset.dataset import PoseSequenceDataset, collate_fn

print("インポート完了")
print(f"PyTorchバージョン: {torch.__version__}")
print(f"CUDAが利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 5. データパスの設定

In [ ]:
# データファイルのパスを設定
data_root = '/content/drive/MyDrive/Visuable_for_you_tabletennis/data/detect'

# 訓練用動画
train_video_names = [
    'sample_video_03_short',
    'sample_video_04_short',
    'sample_video_05_02',
]

# 検証用動画
val_video_names = [
    'sample_video_06_01',
]

# 訓練データのパスを構築（各動画ごとのディレクトリ）
train_data_dirs = [f"{data_root}/{video}" for video in train_video_names]
val_data_dirs = [f"{data_root}/{video}" for video in val_video_names]

# 出力ディレクトリ
OUTPUT_DIR = "output/training"

# ファイルの存在確認
print("=" * 60)
print("データディレクトリの確認")
print("=" * 60)

print("\n[訓練データ]")
for i, video in enumerate(train_video_names):
    data_dir = train_data_dirs[i]
    csv_origin_path = os.path.join(data_dir, "original_pose_data.csv")
    csv_aug_path = os.path.join(data_dir, "augment_pose_data.csv")
    label_path = os.path.join(data_dir, "play_labels.csv")
    
    dir_exists = os.path.exists(data_dir)
    csv_origin_exists = os.path.exists(csv_origin_path)
    csv_aug_exists = os.path.exists(csv_aug_path)
    label_exists = os.path.exists(label_path)
    
    print(f"  {video}:")
    print(f"    Dir: {'✓' if dir_exists else '✗'} {data_dir}")
    print(f"    CSV (Original): {'✓' if csv_origin_exists else '✗'} {csv_origin_path}")
    print(f"    CSV (Augmented): {'✓' if csv_aug_exists else '✗'} {csv_aug_path}")
    print(f"    Label: {'✓' if label_exists else '✗'} {label_path}")

print("\n[検証データ]")
for i, video in enumerate(val_video_names):
    data_dir = val_data_dirs[i]
    csv_origin_path = os.path.join(data_dir, "original_pose_data.csv")
    csv_aug_path = os.path.join(data_dir, "augment_pose_data.csv")
    label_path = os.path.join(data_dir, "play_labels.csv")
    
    dir_exists = os.path.exists(data_dir)
    csv_origin_exists = os.path.exists(csv_origin_path)
    csv_aug_exists = os.path.exists(csv_aug_path)
    label_exists = os.path.exists(label_path)
    
    print(f"  {video}:")
    print(f"    Dir: {'✓' if dir_exists else '✗'} {data_dir}")
    print(f"    CSV (Original): {'✓' if csv_origin_exists else '✗'} {csv_origin_path}")
    print(f"    CSV (Augmented): {'✓' if csv_aug_exists else '✗'} {csv_aug_path}")
    print(f"    Label: {'✓' if label_exists else '✗'} {label_path}")

print("=" * 60)

## 6. ハイパーパラメータの設定

In [ ]:
# ハイパーパラメータ
config = {
    # モデル
    'model_type': 'lstm',  # 'lstm' or 'cnn_lstm'
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.3,
    'no_attention': False,
    
    # データ
    'sequence_length': 30,  # 30フレーム (約1秒)
    'stride': 5,  # シーケンスのストライド
    
    # 学習
    'epochs': 50,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'num_workers': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # その他
    'save_every': 10,
}

print("設定:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 7. データセットの作成

In [ ]:
from torch.utils.data import ConcatDataset

print("データセット作成中...")

# 訓練データセット: 複数動画のデータを結合
train_datasets = []
for i, video_name in enumerate(train_video_names):
    data_dir = train_data_dirs[i]
    csv_path = os.path.join(data_dir, "augment_pose_data.csv")  # 拡張データを使用
    label_path = os.path.join(data_dir, "play_labels.csv")
    
    if not os.path.exists(csv_path):
        print(f"⚠ 拡張データが見つかりません。オリジナルデータを使用: {video_name}")
        csv_path = os.path.join(data_dir, "original_pose_data.csv")
    
    if os.path.exists(csv_path) and os.path.exists(label_path):
        dataset = PoseSequenceDataset(
            csv_path=csv_path,
            label_path=label_path,
            sequence_length=config['sequence_length'],
            stride=config['stride']
        )
        train_datasets.append(dataset)
        print(f"  ✓ {video_name}: {len(dataset)} シーケンス")
    else:
        print(f"  ✗ {video_name}: ファイルが見つかりません")

# 複数のデータセットを結合
if len(train_datasets) == 0:
    raise ValueError("訓練データセットが作成できませんでした")

train_dataset = ConcatDataset(train_datasets)

train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    collate_fn=collate_fn,
    pin_memory=True if config['device'] == 'cuda' else False
)

# 検証データセット: オリジナルデータのみ使用
val_loader = None
if len(val_video_names) > 0:
    val_datasets = []
    for i, video_name in enumerate(val_video_names):
        data_dir = val_data_dirs[i]
        csv_path = os.path.join(data_dir, "original_pose_data.csv")  # オリジナルデータを使用
        label_path = os.path.join(data_dir, "play_labels.csv")
        
        if os.path.exists(csv_path) and os.path.exists(label_path):
            dataset = PoseSequenceDataset(
                csv_path=csv_path,
                label_path=label_path,
                sequence_length=config['sequence_length'],
                stride=config['stride']
            )
            val_datasets.append(dataset)
            print(f"  ✓ {video_name} (Val): {len(dataset)} シーケンス")
        else:
            print(f"  ✗ {video_name} (Val): ファイルが見つかりません")
    
    if len(val_datasets) > 0:
        val_dataset = ConcatDataset(val_datasets)
        val_loader = DataLoader(
            val_dataset,
            batch_size=config['batch_size'],
            shuffle=False,
            num_workers=config['num_workers'],
            collate_fn=collate_fn,
            pin_memory=True if config['device'] == 'cuda' else False
        )
        print(f"\n検証データセット: {len(val_dataset)} シーケンス")

print(f"\n訓練データセット: {len(train_dataset)} シーケンス")
print(f"訓練バッチ数: {len(train_loader)}")

# サンプルデータの確認
sample_features, sample_labels, sample_metadata = train_dataset[0]
print(f"\nサンプルデータ:")
print(f"  特徴量shape: {sample_features.shape}")
print(f"  ラベルshape: {sample_labels.shape}")
print(f"  メタデータ: {sample_metadata}")

## 8. モデルの作成

In [ ]:
print("モデル作成中...")

# モデルの作成
if config['model_type'] == 'lstm':
    model = PlayClassifierLSTM(
        input_size=34,  # 17キーポイント × 2座標
        hidden_size=config['hidden_size'],
        num_layers=config['num_layers'],
        dropout=config['dropout'],
        use_attention=not config['no_attention']
    )
else:  # cnn_lstm
    model = PlayClassifierCNNLSTM(
        input_size=34,
        hidden_size=config['hidden_size'],
        num_layers=config['num_layers'],
        dropout=config['dropout']
    )

# デバイスに転送
device = torch.device(config['device'])
model = model.to(device)

print(f"モデルタイプ: {config['model_type']}")
print(f"パラメータ数: {sum(p.numel() for p in model.parameters()):,}")
print(f"デバイス: {device}")

## 9. トレーニング関数の定義

In [ ]:
class Trainer:
    """モデル学習用クラス"""
    
    def __init__(self, model, train_loader, val_loader, device, learning_rate, output_dir):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # 損失関数
        self.criterion = nn.BCELoss()
        
        # オプティマイザ
        self.optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        
        # 学習率スケジューラ
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=5
        )
        
        # 学習履歴
        self.history = {
            'train_loss': [], 'train_acc': [], 'train_f1': [],
            'val_loss': [], 'val_acc': [], 'val_f1': []
        }
        
        self.best_val_f1 = 0.0
    
    def train_epoch(self, epoch):
        """1エポックの学習"""
        self.model.train()
        total_loss = 0
        all_preds = []
        all_labels = []
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch} [Train]")
        for features, labels, metadata in pbar:
            features = features.to(self.device)
            labels = labels.to(self.device)
            
            # 順伝播
            self.optimizer.zero_grad()
            outputs = self.model(features).squeeze(-1)
            
            # 損失計算
            loss = self.criterion(outputs, labels)
            
            # 逆伝播
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            # 統計
            total_loss += loss.item()
            preds = (outputs > 0.5).float().cpu().detach().numpy()
            labels_np = labels.cpu().detach().numpy()
            all_preds.append(preds.flatten())
            all_labels.append(labels_np.flatten())
            
            pbar.set_postfix({'loss': loss.item()})
        
        # エポック全体の統計
        avg_loss = total_loss / len(self.train_loader)
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        metrics = self._compute_metrics(all_labels, all_preds)
        metrics['loss'] = avg_loss
        
        return metrics
    
    def validate(self, epoch):
        """検証"""
        if self.val_loader is None:
            return {}
        
        self.model.eval()
        total_loss = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc=f"Epoch {epoch} [Val]")
            for features, labels, metadata in pbar:
                features = features.to(self.device)
                labels = labels.to(self.device)
                
                outputs = self.model(features).squeeze(-1)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()
                
                preds = (outputs > 0.5).float().cpu().numpy()
                labels_np = labels.cpu().numpy()
                all_preds.append(preds.flatten())
                all_labels.append(labels_np.flatten())
                
                pbar.set_postfix({'loss': loss.item()})
        
        avg_loss = total_loss / len(self.val_loader)
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        metrics = self._compute_metrics(all_labels, all_preds)
        metrics['loss'] = avg_loss
        
        return metrics
    
    def _compute_metrics(self, labels, preds):
        """評価指標の計算"""
        accuracy = np.mean(labels == preds)
        
        tp = np.sum((labels == 1) & (preds == 1))
        fp = np.sum((labels == 0) & (preds == 1))
        fn = np.sum((labels == 1) & (preds == 0))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    def train(self, num_epochs, save_every=10):
        """学習メインループ"""
        print(f"\n{'='*60}")
        print(f"学習開始")
        print(f"  エポック数: {num_epochs}")
        print(f"  デバイス: {self.device}")
        print(f"{'='*60}\n")
        
        for epoch in range(1, num_epochs + 1):
            # 訓練
            train_metrics = self.train_epoch(epoch)
            self.history['train_loss'].append(train_metrics['loss'])
            self.history['train_acc'].append(train_metrics['accuracy'])
            self.history['train_f1'].append(train_metrics['f1'])
            
            # 検証
            val_metrics = self.validate(epoch)
            if val_metrics:
                self.history['val_loss'].append(val_metrics['loss'])
                self.history['val_acc'].append(val_metrics['accuracy'])
                self.history['val_f1'].append(val_metrics['f1'])
                self.scheduler.step(val_metrics['loss'])
            
            # ログ出力
            print(f"\nEpoch {epoch}/{num_epochs}")
            print(f"  Train - Loss: {train_metrics['loss']:.4f}, "
                  f"Acc: {train_metrics['accuracy']:.4f}, F1: {train_metrics['f1']:.4f}")
            if val_metrics:
                print(f"  Val   - Loss: {val_metrics['loss']:.4f}, "
                      f"Acc: {val_metrics['accuracy']:.4f}, F1: {val_metrics['f1']:.4f}")
            
            # チェックポイント保存
            if epoch % save_every == 0:
                self.save_checkpoint(epoch, f"checkpoint_epoch_{epoch}.pth")
            
            # ベストモデル保存
            if val_metrics and val_metrics['f1'] > self.best_val_f1:
                self.best_val_f1 = val_metrics['f1']
                self.save_checkpoint(epoch, "best_model.pth")
                print(f"  ✓ Best model saved (F1: {self.best_val_f1:.4f})")
        
        # 最終モデル保存
        self.save_checkpoint(num_epochs, "final_model.pth")
        self.save_history()
        
        print(f"\n{'='*60}")
        print(f"学習完了")
        print(f"  Best Val F1: {self.best_val_f1:.4f}")
        print(f"{'='*60}\n")
    
    def save_checkpoint(self, epoch, filename):
        """チェックポイント保存"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'history': self.history,
            'best_val_f1': self.best_val_f1
        }
        save_path = self.output_dir / filename
        torch.save(checkpoint, save_path)
    
    def save_history(self):
        """学習履歴を保存"""
        history_path = self.output_dir / 'training_history.json'
        with open(history_path, 'w') as f:
            json.dump(self.history, f, indent=2)

print("Trainerクラス定義完了")

## 10. トレーニングの実行

In [ ]:
# 出力ディレクトリの作成
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path(OUTPUT_DIR) / timestamp
output_dir.mkdir(parents=True, exist_ok=True)

# 設定の保存
config_path = output_dir / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"出力ディレクトリ: {output_dir}")

# Trainerの作成
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    learning_rate=config['learning_rate'],
    output_dir=str(output_dir)
)

# トレーニング開始
trainer.train(
    num_epochs=config['epochs'],
    save_every=config['save_every']
)

print(f"\n学習済みモデル: {output_dir / 'best_model.pth'}")
print(f"学習履歴: {output_dir / 'training_history.json'}")

## 11. 学習結果の可視化

In [ ]:
# 学習履歴の読み込み
history_path = output_dir / 'training_history.json'
with open(history_path, 'r') as f:
    history = json.load(f)

# プロット
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train')
if history['val_loss']:
    axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train')
if history['val_acc']:
    axes[1].plot(history['val_acc'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['train_f1'], label='Train')
if history['val_f1']:
    axes[2].plot(history['val_f1'], label='Val')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].set_title('Training and Validation F1 Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n学習曲線を保存: {output_dir / 'training_curves.png'}")

## 12. モデルの評価（オプション）

学習済みモデルで検証データを評価します

In [ ]:
# ベストモデルを読み込み
best_model_path = output_dir / 'best_model.pth'
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"ベストモデルを読み込み: {best_model_path}")
print(f"エポック: {checkpoint['epoch']}")
print(f"Best Val F1: {checkpoint['best_val_f1']:.4f}")

# 検証データで評価
if val_loader:
    print("\n検証データで評価中...")
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for features, labels, metadata in tqdm(val_loader, desc="評価中"):
            features = features.to(device)
            outputs = model(features).squeeze(-1).cpu().numpy()
            preds = (outputs > 0.5).astype(int)
            labels_np = labels.numpy()
            
            all_probs.append(outputs.flatten())
            all_preds.append(preds.flatten())
            all_labels.append(labels_np.flatten())
    
    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    
    # メトリクスの計算
    accuracy = np.mean(all_labels == all_preds)
    tp = np.sum((all_labels == 1) & (all_preds == 1))
    fp = np.sum((all_labels == 0) & (all_preds == 1))
    fn = np.sum((all_labels == 1) & (all_preds == 0))
    tn = np.sum((all_labels == 0) & (all_preds == 0))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n評価結果:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"\n混同行列:")
    print(f"  TP: {tp}, FP: {fp}")
    print(f"  FN: {fn}, TN: {tn}")
else:
    print("検証データが利用できません")

## 13. モデルをGoogle Driveに保存

学習済みモデルをGoogle Driveにコピーして保存します

In [ ]:
import shutil

# Google Drive上の保存先
SAVE_TO_DRIVE = f"/content/drive/MyDrive/trained_models/play_classifier/{timestamp}"

# ディレクトリ作成
os.makedirs(SAVE_TO_DRIVE, exist_ok=True)

# ファイルをコピー
files_to_copy = [
    'best_model.pth',
    'final_model.pth',
    'config.json',
    'training_history.json',
    'training_curves.png'
]

for filename in files_to_copy:
    src = output_dir / filename
    if src.exists():
        dst = os.path.join(SAVE_TO_DRIVE, filename)
        shutil.copy2(src, dst)
        print(f"コピー完了: {filename} -> {dst}")

print(f"\nモデルをGoogle Driveに保存: {SAVE_TO_DRIVE}")

## 完了

トレーニングが完了しました！

次のステップ:
1. 学習済みモデルをダウンロード
2. 推論スクリプト (`predict_play_scenes.py`) を使って動画から予測
3. モデルのパフォーマンスを評価し、必要に応じてハイパーパラメータを調整